## Part 1: Word Embeddings
### 1. TF-IDF and PMI Weighted Representations
#### 1.1 TF-IDF Weighting

In [4]:
# Cell 1: Imports & Setup
import numpy as np
import json
import re
from collections import Counter, defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import os

# Create output directories
os.makedirs("embeddings", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

print("Directories ready.")
print(f"NumPy: {np.__version__}")

Directories ready.
NumPy: 2.3.5


#### 1.2 Pointwise Mutual Information (PMI)

In [5]:
# Cell 2: Load cleaned.txt and raw.txt as documents (one line = one document)
def load_documents(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        docs = [line.strip() for line in f if line.strip()]
    return docs

def tokenize(text):
    # Basic whitespace + punctuation tokenizer for Urdu
    return re.findall(r'\S+', text)

cleaned_docs = load_documents("cleaned.txt")
raw_docs     = load_documents("raw.txt")

print(f"cleaned.txt → {len(cleaned_docs)} documents")
print(f"raw.txt     → {len(raw_docs)} documents")
print(f"\nSample doc (cleaned):\n{cleaned_docs[0][:200]}")

cleaned.txt → 500 documents
raw.txt     → 10097 documents

Sample doc (cleaned):
[1]


In [6]:
# Cell 3: Build vocabulary capped at 10,000 tokens + <UNK>
all_tokens = []
for doc in cleaned_docs:
    all_tokens.extend(tokenize(doc))

token_freq = Counter(all_tokens)
print(f"Total unique tokens: {len(token_freq)}")

VOCAB_SIZE = 10_000
top_tokens = [tok for tok, _ in token_freq.most_common(VOCAB_SIZE)]

# Build mappings
word2idx = {tok: idx+1 for idx, tok in enumerate(top_tokens)}  # 0 reserved for <UNK>
word2idx["<UNK>"] = 0
idx2word = {idx: tok for tok, idx in word2idx.items()}

print(f"Vocabulary size (incl. <UNK>): {len(word2idx)}")
print(f"Top 10 tokens: {top_tokens[:10]}")

# Save word2idx for later parts
with open("embeddings/word2idx.json", "w", encoding="utf-8") as f:
    json.dump(word2idx, f, ensure_ascii=False)
print("word2idx.json saved.")

Total unique tokens: 15312
Vocabulary size (incl. <UNK>): 10001
Top 10 tokens: ['کے', 'میں', 'کی', 'ہے', 'اور', 'سے', 'کہ', 'کا', 'کو', 'نے']
word2idx.json saved.


In [7]:
# Cell 4: TF-IDF Matrix
V = len(word2idx)   # 10001
N = len(cleaned_docs)

# --- Term Frequency (raw counts per doc) ---
print("Building term-document count matrix...")
# Use a list of Counters for memory efficiency, then convert
tf_matrix = np.zeros((V, N), dtype=np.float32)

for doc_idx, doc in enumerate(cleaned_docs):
    tokens = tokenize(doc)
    counts = Counter(tokens)
    total  = len(tokens) if len(tokens) > 0 else 1
    for tok, cnt in counts.items():
        widx = word2idx.get(tok, 0)   # 0 = <UNK>
        tf_matrix[widx, doc_idx] += cnt / total   # TF = count/doc_length

    if doc_idx % 500 == 0:
        print(f"  Processed {doc_idx}/{N} docs...")

print("TF matrix built.")

# --- Document Frequency ---
df = np.sum(tf_matrix > 0, axis=1).astype(np.float32)  # shape (V,)

# --- IDF ---
idf = np.log(N / (1 + df))   # shape (V,)

# --- TF-IDF ---
# Broadcast: tfidf[w,d] = tf[w,d] * idf[w]
tfidf_matrix = tf_matrix * idf[:, np.newaxis]

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")  # (10001, N_docs)

# Save
np.save("embeddings/tfidf_matrix.npy", tfidf_matrix)
print("tfidf_matrix.npy saved ✓")

Building term-document count matrix...
  Processed 0/500 docs...
TF matrix built.
TF-IDF matrix shape: (10001, 500)
tfidf_matrix.npy saved ✓


In [8]:
# Cell 5: Top-10 most discriminative words per topic using Metadata.json
with open("Metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Metadata keys: {list(metadata[0].keys()) if isinstance(metadata, list) else list(metadata.keys())}")
print(f"Sample entry: {metadata[0] if isinstance(metadata, list) else list(metadata.items())[:2]}")

Metadata keys: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', 